# GPT entity extraction for three input representations

This notebook extracts entities from Dutch administrative decisions using three input representations:

1. `full_text`
2. `section_based`
3. `sentence_based`

Every document and representation is sent in a **new, independent Responses API request**. The request contains only the fixed prompt and the current text. It does not pass a `previous_response_id`, conversation history, or earlier extraction results. `store=False` is also used.

The notebook saves:

- one checkpoint JSON file per document;
- separate JSONL and CSV result files for each representation;
- the extracted JSON, request status, hashes, timestamps, and model response ID;
- input, cached-input, output, reasoning, and total tokens;
- elapsed API time and retry count;
- estimated cost in USD using editable rates;
- combined results, summaries, errors, and a document-alignment report.

**Before running the API calls:** edit the paths and model settings, preview the discovered files, set `RUN_API = True`, and then run the three extraction cells. The API key is entered securely and is not written to the notebook or output files.

In [1]:
# Run once in the notebook environment.
#!pip install -U openai pydantic pandas tqdm

In [2]:
from __future__ import annotations

import getpass
import hashlib
import json
import os
import random
import re
import time
import traceback
import uuid
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd
from openai import (
    APIConnectionError,
    APITimeoutError,
    InternalServerError,
    OpenAI,
    RateLimitError,
)
from pydantic import BaseModel, ConfigDict
from tqdm.auto import tqdm

pd.set_option("display.max_colwidth", 120)

## 1. Configuration

Change the three input paths to the folders containing your full-text, section-based, and sentence-based files. Supported input formats are `.txt`, `.md`, and several common `.json` structures.

The default model is `gpt-5.6-terra`. Keep the same model and settings for all three representations for a controlled comparison. The USD calculation is an estimate; the token counts returned by the API are the authoritative usage measurements. Update the rates if you change the model, service tier, processing region, or context-pricing category.

In [3]:
# -----------------------------
# Paths: edit these four values
# -----------------------------
PROJECT_DIR = Path(r"E:\\CITaDOG")

INPUT_DIRS = {
    "full_text": Path(r"E:\CITaDOG\sampling_analysis\sample_700_markdown"),
    "section_based": Path(r"E:\\CITaDOG\\sampling_analysis\\sample_700_operative_simple"),
    "sentence_based": Path(r"E:\\CITaDOG\\sampling_analysis\\sentence_selection_updated\\selected_documents")
}

OUTPUT_DIR = PROJECT_DIR / "gpt_entity_extraction_results"

# -----------------------------
# API/model settings
# -----------------------------
MODEL = "gpt-5.6-luna"
REASONING_EFFORT = None      # Set to None to omit this parameter.
MAX_OUTPUT_TOKENS = 4_000
REQUEST_TIMEOUT_SECONDS = 600
MAX_RETRIES = 5

# Files are processed sequentially. This makes timings and checkpoints simple.
FILE_EXTENSIONS = {".txt", ".md", ".json"}
MAX_FILES_PER_INPUT = None       # e.g. 2 for a small paid test; None = all files.
RESUME = True                    # Skip successful unchanged documents.

# Safety switch. Set to True only after checking the file preview below.
RUN_API = False

# Estimated STANDARD, SHORT-CONTEXT prices in USD per 1M tokens,
# recorded from OpenAI's pricing page on 2026-09-02.
# Change these if you change the model, context tier, service tier, or region.
PRICE_RATES_USD_PER_1M = {
    "gpt-5.6-sol":   {"input": 2.00, "cached_input": 0.20, "output": 10.00},
    "gpt-5.6-terra": {"input": 1.00, "cached_input": 0.10, "output": 6.00},
    "gpt-5.6-luna":  {"input": 0.10, "cached_input": 0.01, "output": 0.60},
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
INPUT_DIRS, OUTPUT_DIR

({'full_text': WindowsPath('E:/CITaDOG/sampling_analysis/sample_700_markdown'),
  'section_based': WindowsPath('E:/CITaDOG/sampling_analysis/sample_700_operative_simple'),
  'sentence_based': WindowsPath('E:/CITaDOG/sampling_analysis/sentence_selection_updated/selected_documents')},
 WindowsPath('E:/CITaDOG/gpt_entity_extraction_results'))

## 2. Fixed extraction prompt

The placeholder `{TEXT}` is replaced separately for every API call. The prompt below is saved in the run manifest so the experiment can be reproduced.

In [4]:
PROMPT_TEMPLATE = """Je bent een informatie-extractiesysteem voor Nederlandse bestuursrechtelijke besluiten (*beschikkingen*).

Extraheer de volgende informatie uit de aangeleverde tekst:

1. **Ontvanger**: de persoon, onderneming, organisatie of andere partij van wie de rechtspositie rechtstreeks wordt bepaald door het huidige besluit.
2. **Besluitvormend orgaan**: het bestuursorgaan of de bestuurlijke autoriteit die het huidige besluit neemt.
3. **Rechtshandeling**: de juridisch relevante handeling die door het besluitvormend orgaan wordt verricht, zoals `verleent`, `weigert`, `wijst af`, `legt op`, `trekt in` of `verklaart ongegrond`.
4. **Rechtsobject**: het juridische instrument, verzoek, de sanctie, aanspraak of ander object waarop de rechtshandeling betrekking heeft, zoals `vergunning`, `ontheffing`, `subsidie`, `boete`, `last onder dwangsom`, `verzoek` of `bezwaar`.

### Instructies

- Extraheer alleen informatie die betrekking heeft op het **huidige bestuursrechtelijke besluit**.
- Extraheer geen informatie die betrekking heeft op eerdere besluiten, aanvragen, voorgenomen besluiten, rechterlijke uitspraken, feitelijke achtergrond of handelingen van andere partijen.
- De Rechtshandeling en het Rechtsobject moeten betrekking hebben op **dezelfde beslissing**.
- Geef bij scheidbare werkwoorden de volledige rechtshandeling terug, bijvoorbeeld `wijst af`, `legt op` of `trekt in`.
- Wanneer termen zoals `aanvrager`, `betrokkene`, `overtreder`, `wij` of `u` worden gebruikt, herleid deze dan tot de expliciet genoemde partij indien dit op basis van de aangeleverde tekst mogelijk is.
- Gebruik uitsluitend de aangeleverde tekst. De tekst kan een onvolledig fragment van een groter besluit zijn.
- Als informatie niet uit de aangeleverde tekst kan worden vastgesteld, geef dan `null` of een lege lijst terug zoals hieronder aangegeven.
- Raad niet en gebruik geen externe kennis.
- Behoud namen en rechtsobjecten zoveel mogelijk zoals zij in de tekst voorkomen.

Geef **uitsluitend geldige JSON** terug in het volgende formaat:

{
"ontvanger": [],
"besluitvormend_orgaan": null,
"beslissingen": [
{
"rechtshandeling": null,
"rechtsobject": null
}
]
}

Als geen ontvanger kan worden vastgesteld, geef dan:

"ontvanger": []

Als geen combinatie van Rechtshandeling en Rechtsobject kan worden vastgesteld, geef dan:

"beslissingen": []

Als het huidige besluit meerdere ontvangers of meerdere afzonderlijke combinaties van Rechtshandeling en Rechtsobject bevat, geef deze dan allemaal terug.

TEKST:
{TEXT}"""

PROMPT_SHA256 = hashlib.sha256(PROMPT_TEMPLATE.encode("utf-8")).hexdigest()
print("Prompt SHA-256:", PROMPT_SHA256)
print("Prompt characters (without document text):", len(PROMPT_TEMPLATE.replace("{TEXT}", "")))

Prompt SHA-256: d4d20d2128d4f1004ae0f30e550e31f9af9fd41388fdf60ae781308ab1bec928
Prompt characters (without document text): 2501


## 3. Structured output schema and API client

The Pydantic schema makes the returned extraction machine-readable. Nullable fields remain required keys, matching the requested JSON structure.

In [5]:
class Beslissing(BaseModel):
    model_config = ConfigDict(extra="forbid")

    rechtshandeling: str | None
    rechtsobject: str | None


class ExtractieResultaat(BaseModel):
    model_config = ConfigDict(extra="forbid")

    ontvanger: list[str]
    besluitvormend_orgaan: str | None
    beslissingen: list[Beslissing]


if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass(
        "OpenAI API key (hidden; not saved): "
    )

client = OpenAI(
    timeout=REQUEST_TIMEOUT_SECONDS,
    max_retries=0,  # Retries are logged and handled explicitly below.
)

print("Client ready. API calls are currently", "ENABLED" if RUN_API else "DISABLED")

OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.

## 4. Discover and preview the three datasets

Document IDs are relative paths without the file extension. Therefore, the relative directory structure and stem should match across the three input folders. The preview reports missing folders, duplicate IDs, empty files, and cross-representation overlap before any paid request is made.

In [ ]:
def document_id_for(path: Path, root: Path) -> str:
    return path.relative_to(root).with_suffix("").as_posix()


def discover_files(root: Path) -> list[Path]:
    if not root.exists():
        return []
    files = sorted(
        path for path in root.rglob("*")
        if path.is_file() and path.suffix.lower() in FILE_EXTENSIONS
    )
    ids: dict[str, list[Path]] = {}
    for path in files:
        ids.setdefault(document_id_for(path, root), []).append(path)
    duplicates = {doc_id: paths for doc_id, paths in ids.items() if len(paths) > 1}
    if duplicates:
        example = next(iter(duplicates.items()))
        raise ValueError(
            "Duplicate document ID caused by files with the same relative stem: "
            f"{example[0]} -> {example[1]}"
        )
    if MAX_FILES_PER_INPUT is not None:
        files = files[:MAX_FILES_PER_INPUT]
    return files


discovered = {name: discover_files(path) for name, path in INPUT_DIRS.items()}

preview_rows = []
id_sets = {}
for name, root in INPUT_DIRS.items():
    files = discovered[name]
    ids = {document_id_for(path, root) for path in files}
    id_sets[name] = ids
    preview_rows.append({
        "input_representation": name,
        "folder_exists": root.exists(),
        "files_found": len(files),
        "first_file": str(files[0]) if files else None,
    })

display(pd.DataFrame(preview_rows))

all_ids = set().union(*id_sets.values()) if id_sets else set()
alignment_preview = pd.DataFrame({
    "document_id": sorted(all_ids),
    **{name: [doc_id in ids for doc_id in sorted(all_ids)] for name, ids in id_sets.items()},
})

if not alignment_preview.empty:
    display(alignment_preview[list(INPUT_DIRS)].value_counts().rename("documents").reset_index())
else:
    print("No supported input files found. Edit INPUT_DIRS above.")

## 5. Extraction and saving functions

Each request is stateless: `input` is newly constructed from the fixed prompt plus exactly one document; `previous_response_id` is never supplied. A successful unchanged checkpoint is skipped when `RESUME=True`. Changed source text is processed again because its SHA-256 hash no longer matches.

In [ ]:
def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def extract_strings_from_json(data: Any) -> list[str]:
    """Read common sentence/section JSON layouts without including metadata."""
    if isinstance(data, str):
        return [data]
    if isinstance(data, list):
        parts: list[str] = []
        for item in data:
            if isinstance(item, str):
                parts.append(item)
            elif isinstance(item, dict):
                for key in ("text", "sentence", "content", "selected_text"):
                    value = item.get(key)
                    if isinstance(value, str):
                        parts.append(value)
                        break
        return parts
    if isinstance(data, dict):
        for key in (
            "text", "content", "selected_text", "operative_text",
            "section_text", "sentence_text",
        ):
            value = data.get(key)
            if isinstance(value, str):
                return [value]
        for key in (
            "sentences", "selected_sentences", "sections", "items", "results",
        ):
            if key in data:
                parts = extract_strings_from_json(data[key])
                if parts:
                    return parts
    return []


def read_input_text(path: Path) -> str:
    if path.suffix.lower() in {".txt", ".md"}:
        text = path.read_text(encoding="utf-8-sig")
    elif path.suffix.lower() == ".json":
        data = json.loads(path.read_text(encoding="utf-8-sig"))
        parts = extract_strings_from_json(data)
        if not parts:
            raise ValueError(
                "Could not find text in JSON. Expected a string, a list of strings, "
                "or keys such as text/content/sentences/selected_sentences."
            )
        text = "\n\n".join(part.strip() for part in parts if part.strip())
    else:
        raise ValueError(f"Unsupported extension: {path.suffix}")

    text = text.strip()
    if not text:
        raise ValueError("Input file is empty.")
    return text


def safe_checkpoint_name(document_id: str) -> str:
    readable = re.sub(r"[^A-Za-z0-9._-]+", "_", document_id).strip("_")[:100]
    digest = hashlib.sha256(document_id.encode("utf-8")).hexdigest()[:12]
    return f"{digest}__{readable or 'document'}.json"


def checkpoint_path(representation: str, document_id: str) -> Path:
    folder = OUTPUT_DIR / representation / "records"
    folder.mkdir(parents=True, exist_ok=True)
    return folder / safe_checkpoint_name(document_id)


def save_json_atomic(path: Path, data: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(data, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    temporary.replace(path)


def load_checkpoints(representation: str) -> list[dict[str, Any]]:
    folder = OUTPUT_DIR / representation / "records"
    if not folder.exists():
        return []
    records = []
    for path in sorted(folder.glob("*.json")):
        try:
            records.append(json.loads(path.read_text(encoding="utf-8")))
        except Exception as exc:
            print(f"Warning: could not read checkpoint {path.name}: {exc}")
    return records


def usage_value(obj: Any, *attributes: str, default: int = 0) -> int:
    current = obj
    for attribute in attributes:
        if current is None:
            return default
        current = getattr(current, attribute, None)
    return default if current is None else int(current)


def calculate_estimated_cost_usd(
    model: str,
    input_tokens: int,
    cached_input_tokens: int,
    output_tokens: int,
) -> float | None:
    rates = PRICE_RATES_USD_PER_1M.get(model)
    if rates is None:
        return None
    uncached_input_tokens = max(0, input_tokens - cached_input_tokens)
    cost = (
        uncached_input_tokens * rates["input"]
        + cached_input_tokens * rates["cached_input"]
        + output_tokens * rates["output"]
    ) / 1_000_000
    return round(cost, 10)


def response_refusal_text(response: Any) -> str | None:
    refusals = []
    for output_item in getattr(response, "output", []) or []:
        for content_item in getattr(output_item, "content", []) or []:
            refusal = getattr(content_item, "refusal", None)
            if refusal:
                refusals.append(str(refusal))
    return "\n".join(refusals) if refusals else None


TRANSIENT_ERRORS = (
    RateLimitError,
    APIConnectionError,
    APITimeoutError,
    InternalServerError,
)


def call_model_independently(text: str) -> tuple[Any, int, float]:
    """One document, one independent request, no prior response or history."""
    request_input = [
        {
            "role": "user",
            "content": PROMPT_TEMPLATE.replace("{TEXT}", text),
        }
    ]

    request_kwargs: dict[str, Any] = {
        "model": MODEL,
        "input": request_input,
        "text_format": ExtractieResultaat,
        "max_output_tokens": MAX_OUTPUT_TOKENS,
        "store": False,
    }
    if REASONING_EFFORT is not None:
        request_kwargs["reasoning"] = {"effort": REASONING_EFFORT}

    started = time.perf_counter()
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = client.responses.parse(**request_kwargs)
            return response, attempt, time.perf_counter() - started
        except TRANSIENT_ERRORS:
            if attempt == MAX_RETRIES:
                raise
            delay = min(60.0, (2 ** (attempt - 1)) + random.random())
            time.sleep(delay)

    raise RuntimeError("Retry loop ended unexpectedly.")


def record_to_csv_row(record: dict[str, Any]) -> dict[str, Any]:
    extraction = record.get("extraction") or {}
    row = {key: value for key, value in record.items() if key != "extraction"}
    row["ontvanger"] = json.dumps(extraction.get("ontvanger", []), ensure_ascii=False)
    row["besluitvormend_orgaan"] = extraction.get("besluitvormend_orgaan")
    row["beslissingen"] = json.dumps(extraction.get("beslissingen", []), ensure_ascii=False)
    return row


def write_representation_exports(representation: str) -> pd.DataFrame:
    records = sorted(load_checkpoints(representation), key=lambda item: item["document_id"])
    folder = OUTPUT_DIR / representation
    folder.mkdir(parents=True, exist_ok=True)

    jsonl_path = folder / "results.jsonl"
    jsonl_text = "".join(
        json.dumps(record, ensure_ascii=False) + "\n" for record in records
    )
    jsonl_path.write_text(jsonl_text, encoding="utf-8")

    dataframe = pd.DataFrame(record_to_csv_row(record) for record in records)
    dataframe.to_csv(folder / "results.csv", index=False, encoding="utf-8-sig")
    return dataframe


def save_run_manifest() -> None:
    manifest = {
        "created_at_utc": utc_now_iso(),
        "model": MODEL,
        "reasoning_effort": REASONING_EFFORT,
        "max_output_tokens": MAX_OUTPUT_TOKENS,
        "input_dirs": {key: str(value) for key, value in INPUT_DIRS.items()},
        "output_dir": str(OUTPUT_DIR),
        "file_extensions": sorted(FILE_EXTENSIONS),
        "max_files_per_input": MAX_FILES_PER_INPUT,
        "resume": RESUME,
        "prompt_template": PROMPT_TEMPLATE,
        "prompt_sha256": PROMPT_SHA256,
        "price_rates_usd_per_1m": PRICE_RATES_USD_PER_1M,
        "price_note": (
            "Estimated standard short-context rates recorded 2026-09-02; "
            "verify current rates and applicable context/service/region tier."
        ),
        "independence_note": (
            "Each API request contains only the fixed prompt and current document. "
            "No previous_response_id, prior messages, or earlier outputs are supplied; store=False."
        ),
    }
    save_json_atomic(OUTPUT_DIR / "run_manifest.json", manifest)


def run_representation(representation: str) -> pd.DataFrame:
    if representation not in INPUT_DIRS:
        raise KeyError(f"Unknown representation: {representation}")
    if not RUN_API:
        print(f"Skipped {representation}: set RUN_API = True in the configuration cell.")
        return write_representation_exports(representation)

    root = INPUT_DIRS[representation]
    files = discover_files(root)
    if not files:
        raise FileNotFoundError(f"No supported input files found in: {root}")

    save_run_manifest()
    existing = {record["document_id"]: record for record in load_checkpoints(representation)}
    run_id = str(uuid.uuid4())
    run_started_iso = utc_now_iso()
    run_started = time.perf_counter()

    for path in tqdm(files, desc=representation):
        document_id = document_id_for(path, root)
        started_at = utc_now_iso()
        document_started = time.perf_counter()
        text: str | None = None
        try:
            text = read_input_text(path)
            text_hash = sha256_text(text)

            previous = existing.get(document_id)
            if (
                RESUME
                and previous
                and previous.get("status") == "success"
                and previous.get("text_sha256") == text_hash
                and previous.get("model") == MODEL
                and previous.get("prompt_sha256") == PROMPT_SHA256
                and previous.get("reasoning_effort") == REASONING_EFFORT
                and previous.get("max_output_tokens") == MAX_OUTPUT_TOKENS
            ):
                continue

            response, attempts, elapsed_seconds = call_model_independently(text)
            usage = getattr(response, "usage", None)
            input_tokens = usage_value(usage, "input_tokens")
            cached_input_tokens = usage_value(
                usage, "input_tokens_details", "cached_tokens"
            )
            output_tokens = usage_value(usage, "output_tokens")
            reasoning_tokens = usage_value(
                usage, "output_tokens_details", "reasoning_tokens"
            )
            total_tokens = usage_value(
                usage,
                "total_tokens",
                default=input_tokens + output_tokens,
            )

            parsed = getattr(response, "output_parsed", None)
            if parsed is None:
                raise ValueError(
                    "The API returned no parsed extraction. "
                    f"Refusal: {response_refusal_text(response)!r}; "
                    f"raw output: {getattr(response, 'output_text', '')!r}"
                )

            record = {
                "run_id": run_id,
                "document_id": document_id,
                "input_representation": representation,
                "input_file": str(path),
                "relative_input_file": path.relative_to(root).as_posix(),
                "text_characters": len(text),
                "text_sha256": text_hash,
                "prompt_sha256": PROMPT_SHA256,
                "model": MODEL,
                "reasoning_effort": REASONING_EFFORT,
                "max_output_tokens": MAX_OUTPUT_TOKENS,
                "status": "success",
                "started_at_utc": started_at,
                "finished_at_utc": utc_now_iso(),
                "api_elapsed_seconds": round(elapsed_seconds, 6),
                "document_elapsed_seconds": round(time.perf_counter() - document_started, 6),
                "attempts": attempts,
                "response_id": getattr(response, "id", None),
                "response_status": getattr(response, "status", None),
                "response_model": getattr(response, "model", None),
                "input_tokens": input_tokens,
                "cached_input_tokens": cached_input_tokens,
                "uncached_input_tokens": max(0, input_tokens - cached_input_tokens),
                "output_tokens": output_tokens,
                "reasoning_tokens": reasoning_tokens,
                "total_tokens": total_tokens,
                "estimated_cost_usd": calculate_estimated_cost_usd(
                    MODEL,
                    input_tokens,
                    cached_input_tokens,
                    output_tokens,
                ),
                "extraction": parsed.model_dump(),
                "error_type": None,
                "error_message": None,
            }
        except Exception as exc:
            record = {
                "run_id": run_id,
                "document_id": document_id,
                "input_representation": representation,
                "input_file": str(path),
                "relative_input_file": path.relative_to(root).as_posix(),
                "text_characters": len(text) if text is not None else None,
                "text_sha256": sha256_text(text) if text is not None else None,
                "prompt_sha256": PROMPT_SHA256,
                "model": MODEL,
                "reasoning_effort": REASONING_EFFORT,
                "max_output_tokens": MAX_OUTPUT_TOKENS,
                "status": "error",
                "started_at_utc": started_at,
                "finished_at_utc": utc_now_iso(),
                "api_elapsed_seconds": None,
                "document_elapsed_seconds": round(time.perf_counter() - document_started, 6),
                "attempts": None,
                "response_id": None,
                "response_status": None,
                "response_model": None,
                "input_tokens": 0,
                "cached_input_tokens": 0,
                "uncached_input_tokens": 0,
                "output_tokens": 0,
                "reasoning_tokens": 0,
                "total_tokens": 0,
                "estimated_cost_usd": 0.0,
                "extraction": None,
                "error_type": type(exc).__name__,
                "error_message": str(exc),
                "traceback": traceback.format_exc(),
            }

        save_json_atomic(checkpoint_path(representation, document_id), record)
        existing[document_id] = record
        write_representation_exports(representation)

    dataframe = write_representation_exports(representation)
    run_summary = {
        "run_id": run_id,
        "input_representation": representation,
        "run_started_at_utc": run_started_iso,
        "run_finished_at_utc": utc_now_iso(),
        "run_elapsed_seconds": round(time.perf_counter() - run_started, 6),
        "files_discovered": len(files),
        "successful_records": int((dataframe.get("status") == "success").sum()),
        "error_records": int((dataframe.get("status") == "error").sum()),
    }
    save_json_atomic(OUTPUT_DIR / representation / "latest_run_summary.json", run_summary)
    return dataframe


print("Functions loaded.")

## 6. Run each representation separately

Set `RUN_API = True` in the configuration cell first. These are deliberately separate cells. You may initially set `MAX_FILES_PER_INPUT = 2` to verify the complete pipeline with a small paid test. Successful checkpoints are reused when you later change the limit to `None`.

In [ ]:
# 6A. Full-text inputs only
full_text_results = run_representation("full_text")
display(full_text_results.tail())

In [ ]:
# 6B. Section-based inputs only
section_based_results = run_representation("section_based")
display(section_based_results.tail())

In [ ]:
# 6C. Sentence-based inputs only
sentence_based_results = run_representation("sentence_based")
display(sentence_based_results.tail())

## 7. Combine results and create usage summaries

Run this after the extraction cells. It reads the saved checkpoints, not notebook memory, and produces combined JSONL/CSV files, an errors file, a document-alignment report, and totals per representation.

In [ ]:
def build_combined_outputs() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    all_records: list[dict[str, Any]] = []
    for representation in INPUT_DIRS:
        all_records.extend(load_checkpoints(representation))

    all_records = sorted(
        all_records,
        key=lambda item: (item["document_id"], item["input_representation"]),
    )

    combined_jsonl = OUTPUT_DIR / "combined_results.jsonl"
    combined_jsonl.write_text(
        "".join(json.dumps(record, ensure_ascii=False) + "\n" for record in all_records),
        encoding="utf-8",
    )

    combined = pd.DataFrame(record_to_csv_row(record) for record in all_records)
    combined.to_csv(OUTPUT_DIR / "combined_results.csv", index=False, encoding="utf-8-sig")

    if combined.empty:
        summary = pd.DataFrame()
        errors = pd.DataFrame()
        alignment = pd.DataFrame()
    else:
        summary = (
            combined.groupby("input_representation", dropna=False)
            .agg(
                records=("document_id", "size"),
                successful=("status", lambda series: int((series == "success").sum())),
                errors=("status", lambda series: int((series == "error").sum())),
                input_tokens=("input_tokens", "sum"),
                cached_input_tokens=("cached_input_tokens", "sum"),
                output_tokens=("output_tokens", "sum"),
                reasoning_tokens=("reasoning_tokens", "sum"),
                total_tokens=("total_tokens", "sum"),
                total_api_seconds=("api_elapsed_seconds", "sum"),
                mean_api_seconds=("api_elapsed_seconds", "mean"),
                estimated_cost_usd=("estimated_cost_usd", "sum"),
            )
            .reset_index()
        )
        for column in ("total_api_seconds", "mean_api_seconds", "estimated_cost_usd"):
            summary[column] = summary[column].round(6)

        totals = {
            "input_representation": "ALL",
            "records": int(summary["records"].sum()),
            "successful": int(summary["successful"].sum()),
            "errors": int(summary["errors"].sum()),
            "input_tokens": int(summary["input_tokens"].sum()),
            "cached_input_tokens": int(summary["cached_input_tokens"].sum()),
            "output_tokens": int(summary["output_tokens"].sum()),
            "reasoning_tokens": int(summary["reasoning_tokens"].sum()),
            "total_tokens": int(summary["total_tokens"].sum()),
            "total_api_seconds": round(float(summary["total_api_seconds"].sum()), 6),
            "mean_api_seconds": round(float(combined["api_elapsed_seconds"].mean()), 6),
            "estimated_cost_usd": round(float(summary["estimated_cost_usd"].sum()), 6),
        }
        summary = pd.concat([summary, pd.DataFrame([totals])], ignore_index=True)

        errors = combined.loc[
            combined["status"] == "error",
            [
                "document_id", "input_representation", "input_file",
                "error_type", "error_message",
            ],
        ].copy()

        successful = combined.loc[combined["status"] == "success"]
        presence = (
            successful.assign(present=True)
            .pivot_table(
                index="document_id",
                columns="input_representation",
                values="present",
                aggfunc="any",
                fill_value=False,
            )
            .reindex(columns=list(INPUT_DIRS), fill_value=False)
            .reset_index()
        )
        alignment = presence

    summary.to_csv(OUTPUT_DIR / "usage_summary_by_input.csv", index=False, encoding="utf-8-sig")
    errors.to_csv(OUTPUT_DIR / "errors.csv", index=False, encoding="utf-8-sig")
    alignment.to_csv(OUTPUT_DIR / "document_alignment.csv", index=False, encoding="utf-8-sig")
    save_json_atomic(
        OUTPUT_DIR / "usage_summary_by_input.json",
        summary.to_dict(orient="records"),
    )
    return combined, summary, alignment


combined_results, usage_summary, document_alignment = build_combined_outputs()
display(usage_summary)
print("\nSaved outputs to:", OUTPUT_DIR)

## 8. Inspect extractions and failures

These cells do not make API calls.

In [ ]:
if not combined_results.empty:
    display(
        combined_results[
            [
                "document_id", "input_representation", "status",
                "ontvanger", "besluitvormend_orgaan", "beslissingen",
                "total_tokens", "api_elapsed_seconds", "estimated_cost_usd",
            ]
        ].head(20)
    )

    failures = combined_results.loc[combined_results["status"] == "error"]
    print(f"Failures: {len(failures)}")
    if not failures.empty:
        display(failures[["document_id", "input_representation", "error_type", "error_message"]])
else:
    print("No saved results yet.")

## Output layout

```text
gpt_entity_extraction_results/
├── run_manifest.json
├── combined_results.jsonl
├── combined_results.csv
├── usage_summary_by_input.json
├── usage_summary_by_input.csv
├── document_alignment.csv
├── errors.csv
├── full_text/
│   ├── records/*.json
│   ├── results.jsonl
│   └── results.csv
├── section_based/
│   ├── records/*.json
│   ├── results.jsonl
│   └── results.csv
└── sentence_based/
    ├── records/*.json
    ├── results.jsonl
    └── results.csv
```

`total_tokens` is normally `input_tokens + output_tokens`. `reasoning_tokens` is reported separately for analysis but is already included within output usage; do not add it to `total_tokens` again. The estimated cost uses uncached input, cached input, and output token rates. The OpenAI billing dashboard remains authoritative for billed cost.

## API references

- [Structured model outputs](https://developers.openai.com/api/docs/guides/structured-outputs)
- [Text generation with the Responses API](https://developers.openai.com/api/docs/guides/text?api-mode=responses)
- [OpenAI API pricing](https://developers.openai.com/api/docs/pricing)

Review the current pricing page immediately before a large run if the estimated USD cost is important.